In [34]:
import sys
sys.path.append("../src")

import pandas as pd
from IPython.display import display, Markdown

from tickers import WATCHLIST, OWNED, sanitize_tickers
from parameters import (
    ACCOUNT_SIZE,
    RISK_PER_TRADE,
    WATCHLIST_ROWS_TO_SHOW,
    OWNED_ROWS_TO_SHOW,
)
from data_fetch import prefetch_prices
from strategy import run_prefetched


# Sanitize tickers
watchlist_clean, watch_remap, watch_drop = sanitize_tickers(WATCHLIST)
owned_clean, owned_remap, owned_drop = sanitize_tickers(OWNED)

dupes_watch = sorted({t for t in WATCHLIST if WATCHLIST.count(t) > 1})
dupes_owned = sorted({t for t in OWNED if OWNED.count(t) > 1})

if watch_remap or watch_drop or dupes_watch:
    print(
        "WATCHLIST → remapped:", watch_remap,
        "| dropped:", sorted(set(watch_drop)),
        "| duplicates removed:", dupes_watch
    )

if owned_remap or owned_drop or dupes_owned:
    print(
        "OWNED     → remapped:", owned_remap,
        "| dropped:", sorted(set(owned_drop)),
        "| duplicates removed:", dupes_owned
    )


# Fetch and score
print("Fetching prices...")
watch_cache = prefetch_prices(watchlist_clean)
owned_cache = prefetch_prices(owned_clean)

print("Scoring...")
df_watch = run_prefetched(watch_cache, ACCOUNT_SIZE, RISK_PER_TRADE)
df_owned = run_prefetched(owned_cache, ACCOUNT_SIZE, RISK_PER_TRADE)


# Display results
BASE_COLS = [
    "Ticker", "BuyScore", "SellUrgency", "RSI14", "ATR%",
    "R_Ratio", "Stop", "Target", "SuggestedShares", "Notes"
]

def show_group(name: str, df: pd.DataFrame, n=None, sort_alpha: bool = False, sorted_limit: int = 20):
    cols = [c for c in BASE_COLS if c in df.columns]
    row_cap = n if n is not None else len(df)
    disp_cap = sorted_limit

    label = "all" if n is None else f"top {row_cap}"
    df_raw = df.sort_values("Ticker").reset_index(drop=True) if sort_alpha else df
    display(Markdown(f"## {name}: Raw ({label})"))
    display(df_raw[cols].head(row_cap))

    for sort_col, ascending, label in [
        ("BuyScore", False, f"Top {disp_cap} by BuyScore"),
        ("SellUrgency", False, f"Top {disp_cap} by SellUrgency"),
        ("ATR%", True, f"Lowest {disp_cap} by ATR%"),
    ]:
        display(Markdown(f"### {name}: {label}"))
        display(
            df.sort_values(sort_col, ascending=ascending)
              .reset_index(drop=True)[cols]
              .head(disp_cap)
        )

show_group("WATCHLIST", df_watch, n=WATCHLIST_ROWS_TO_SHOW, sort_alpha=True)
show_group("OWNED", df_owned, n=OWNED_ROWS_TO_SHOW, sort_alpha=False)

WATCHLIST → remapped: {} | dropped: [] | duplicates removed: ['URA']
Fetching prices...



4 Failed downloads:
['SPR', 'PLL', 'LTHM', 'K']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')


Scoring...


## WATCHLIST: Raw (all)

,Ticker,BuyScore,SellUrgency,RSI14,ATR%,R_Ratio,Stop,Target,SuggestedShares,Notes
0,AA,51,0,60.5,6.64,1.50,60.72,75.44,10,Close>SMA50 | SMA20>SMA50 | MACD>0 | RSI healt...
1,AAL,10,65,24.1,6.54,-2.00,11.71,12.93,6000,"RSI weak (<40) | Volatile | Liquid (≥$5,000,00..."
2,AAPL,23,55,23.7,2.26,-80.54,252.99,266.15,6000,SMA20>SMA50 | RSI weak (<40) | ATR% in sweet s...
3,ABBV,51,20,40.2,2.68,10.51,219.96,237.11,40,SMA20>SMA50 | Pullback to EMA20 | ATR% in swee...
4,ABNB,68,20,54.0,3.80,5.16,126.25,139.00,28,SMA20>SMA50 | MACD>0 | RSI healthy (50–70) | P...
...,...,...,...,...,...,...,...,...,...,...
473,YETI,10,65,11.1,4.76,-2.21,40.19,44.20,6000,"RSI weak (<40) | Volatile | Liquid (≥$5,000,00..."
474,YUM,43,0,41.0,2.26,1.18,155.60,169.09,9,Close>SMA50 | SMA20>SMA50 | MACD>0 | ATR% in s...
475,ZBH,54,20,31.5,2.88,3.30,91.68,98.67,36,Close>SMA50 | SMA20>SMA50 | RSI weak (<40) | P...
476,ZM,10,55,25.1,4.64,-3.44,76.98,81.92,6000,"RSI weak (<40) | Volatile | Liquid (≥$5,000,00..."


...

,Ticker,BuyScore,SellUrgency,RSI14,ATR%,R_Ratio,Stop,Target,SuggestedShares,Notes
0,MP,48,20,51.9,6.28,5.71,58.05,66.81,45,RSI healthy (50–70) | Pullback to EMA20 | Vola...
1,UNH,48,20,61.8,2.86,1.95,277.11,301.84,7,RSI healthy (50–70) | Pullback to EMA20 | ATR%...
2,OPEN,38,0,51.3,7.09,2.73,4.91,5.91,222,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."
3,AMD,33,20,37.8,4.34,123.94,196.44,213.65,435,RSI weak (<40) | Pullback to EMA20 | Volatile ...
4,HOOD,25,55,52.6,6.15,-7.93,76.50,84.75,6000,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."
5,SOFI,15,65,41.2,5.72,-2.65,18.39,19.65,6000,"Volatile | Liquid (≥$5,000,000) | R:R thin (-2..."


### OWNED: Top 20 by SellUrgency

,Ticker,BuyScore,SellUrgency,RSI14,ATR%,R_Ratio,Stop,Target,SuggestedShares,Notes
0,SOFI,15,65,41.2,5.72,-2.65,18.39,19.65,6000,"Volatile | Liquid (≥$5,000,000) | R:R thin (-2..."
1,HOOD,25,55,52.6,6.15,-7.93,76.50,84.75,6000,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."
2,MP,48,20,51.9,6.28,5.71,58.05,66.81,45,RSI healthy (50–70) | Pullback to EMA20 | Vola...
3,UNH,48,20,61.8,2.86,1.95,277.11,301.84,7,RSI healthy (50–70) | Pullback to EMA20 | ATR%...
4,AMD,33,20,37.8,4.34,123.94,196.44,213.65,435,RSI weak (<40) | Pullback to EMA20 | Volatile ...
5,OPEN,38,0,51.3,7.09,2.73,4.91,5.91,222,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."


### OWNED: Lowest 20 by ATR%

,Ticker,BuyScore,SellUrgency,RSI14,ATR%,R_Ratio,Stop,Target,SuggestedShares,Notes
0,UNH,48,20,61.8,2.86,1.95,277.11,301.84,7,RSI healthy (50–70) | Pullback to EMA20 | ATR%...
1,AMD,33,20,37.8,4.34,123.94,196.44,213.65,435,RSI weak (<40) | Pullback to EMA20 | Volatile ...
2,SOFI,15,65,41.2,5.72,-2.65,18.39,19.65,6000,"Volatile | Liquid (≥$5,000,000) | R:R thin (-2..."
3,HOOD,25,55,52.6,6.15,-7.93,76.50,84.75,6000,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."
4,MP,48,20,51.9,6.28,5.71,58.05,66.81,45,RSI healthy (50–70) | Pullback to EMA20 | Vola...
5,OPEN,38,0,51.3,7.09,2.73,4.91,5.91,222,"RSI healthy (50–70) | Volatile | Liquid (≥$5,0..."
